# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



## Setup: Create Agent

In [1]:
import os
os.environ["LOG_LEVEL"] = "WARNING"

In [1]:
from gai.lib.constants import DEFAULT_GUID
from gai.asm.agents import ToolUseAgent2
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.messages import FileMonologue
from gai.messages import FileDialogue, MessagePydantic

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()

def print_chunk(chunk):
    """
    Helper function to print the chunk of data received from the agent.
    """
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
        else:
            if isinstance(chunk, list):
                for item in chunk:
                    if item.get("name"):
                        print(f'Tool: "{item["name"]}"')
                    if item.get("input"):
                        inputs = item.get("input")
                        if isinstance(inputs, dict):
                            for key, value in inputs.items():
                                if isinstance(value, str):
                                    if len(value) > 100:
                                        print(
                                            f"\tInput: {key} = {value[:100]}... (truncated)"
                                        )
                                    else:
                                        print(f"\tInput: {key} = {value}")

---

## Scenario 1: Context inferred from dialogue recap

In this scenario, we create a hypothetical conversation where the user talks about weather in Singapore before asking for the time.

This is to demonstrate that the agent can infer that the user is asking for current time in Singapore.

If this doesn't work, the agent will ask the user to clarify the location.


### a) Arrange

In [2]:
# Create an artificial dialogue history for testing

messages = [
    MessagePydantic(
        **{
            "id": "b1e5f98c-f6eb-47de-a6e2-387510d970f9",
            "header": {
                "sender": "User",
                "recipient": "Sara",
                "timestamp": 1751308157.270983,
                "order": 0,
            },
            "body": {
                "type": "chat.send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 0,
                "role": "user",
                "content": "It is a very nice weather in Singapore right now.",
            },
        }
    ),
    MessagePydantic(
        **{
            "id": "abbc7961-45dc-4973-aaf4-a6224ed35d37",
            "header": {
                "sender": "Sara",
                "recipient": "User",
                "timestamp": 1751308167.3488164,
                "order": 1,
            },
            "body": {
                "type": "chat.reply",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 1,
                "chunk_no": 10,
                "chunk": "<eom>",
                "role": "assistant",
                "content": "Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?",
            },
        }
    ),
]
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages, file_path=file_path)

agent = ToolUseAgent2(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue
)

recap = dialogue.extract_recap()

# Update dialogue

user_message = "What is the current time here?"
dialogue.add_user_message(recipient="Sara", content=user_message)

# INIT
print(f"\ncompleted state: {agent.fsm.state}")

# INIT -> IS_TOOL_CALL
resp = await agent.start_async()
assert agent.fsm.state == "IS_TOOL_CALL"
print(f"\ncompleted state: {agent.fsm.state}")
print(f"\nis_tool_call_result: {agent.fsm.state_bag['is_tool_call_result']}")
print(f"\npredicate_result: {agent.fsm.state_bag['predicate_result']}")

# IS_TOOL_CALL -> CHAT
resp = await agent.resume_async(user_message=user_message, recap=recap)
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "CHAT"

# CHAT -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"
assert agent.fsm.state_bag["is_terminate_result"] is False
assert agent.fsm.state_bag["predicate_result"] is False




completed state: INIT

completed state: IS_TOOL_CALL

is_tool_call_result: False

predicate_result: False
I'll help you get the current time in Singapore. Let me fetch that information for you.
Tool: "current_time"
	Input: format = YYYY-MM-DD HH:mm:ss
	Input: timezone = Asia/Singapore

completed state: CHAT

completed state: IS_TERMINATE


### b) Act

In [3]:

# IS_TERMINATE -> IS_TOOL_CALL
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TOOL_CALL"
assert agent.fsm.state_bag["is_tool_call_result"] is True
assert agent.fsm.state_bag["predicate_result"] is True

# IS_TOOL_CALL -> TOOL_USE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "TOOL_USE"

# TOOL_USE -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"

assistant_message = agent.fsm.state_bag["get_assistant_message"]()
dialogue.add_assistant_message(sender="Sara", chunk="<eom>",content=assistant_message)



completed state: IS_TOOL_CALL
The current time in Singapore is **July 29, 2025 at 4:45:47 AM**. It's quite early in the morning! I hope you're enjoying the nice weather you mentioned.

completed state: TOOL_USE

completed state: IS_TERMINATE


MessagePydantic(id='b719a2b8-4300-46b4-b75f-d5bced8b8c5c', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753735556.387109, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.33', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="The current time in Singapore is **July 29, 2025 at 4:45:47 AM**. It's quite early in the morning! I hope you're enjoying the nice weather you mentioned."))

### c) Show monologue

See the monologue for details at `tmp/monologue.json`

In [7]:
import json
from gai.messages import message_helper

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_chat_messages()
for message in messages[-2:]:
    print(json.dumps(message, indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size=message_helper.get_messages_length(messages)
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": "toolu_01MFNgZZS4JGotWB5xr168k6",
            "content": "Current UTC time is 2025-07-28 20:45:47, and the time in Asia/Singapore is 2025-07-29 04:45:47."
        }
    ]
}
{
    "role": "assistant",
    "content": [
        {
            "citations": null,
            "text": "The current time in Singapore is **July 29, 2025 at 4:45:47 AM**. It's quite early in the morning! I hope you're enjoying the nice weather you mentioned.",
            "type": "text"
        }
    ]
}
───────────────────────── MONOLOGUE END ─────────────────────────

Total char size= 2024


### d) Show dialogue

In [5]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: It is a very nice weather in Singapore right now.
Sara: Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?
User: Sara, What is the current time here?
Sara: The current time in Singapore is **July 29, 2025 at 4:45:47 AM**. It's quite early in the morning! I hope you're enjoying the nice weather you mentioned.


---

## Scenario 2: Agent interrupt user for input

In this scenario, we will not use dialogue recap.

We just ask the agent directly about the time without giving any context. The agent should ask the user for input to continue the conversation.

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

### a) arrange

In [2]:
aggregated_client = McpAggregatedClient(["mcp-pseudo", "mcp-time", "mcp-web"])
agent = ToolUseAgent2(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
)

user_message = "When is the next public holiday? Please ask if you need more information."

# INIT
print(f"\ncompleted state: {agent.fsm.state}")

# INIT -> IS_TOOL_CALL
resp = await agent.start_async()
assert agent.fsm.state == "IS_TOOL_CALL"
print(f"\ncompleted state: {agent.fsm.state}")
print(f"\nis_tool_call_result: {agent.fsm.state_bag['is_tool_call_result']}")
print(f"\npredicate_result: {agent.fsm.state_bag['predicate_result']}")

# IS_TOOL_CALL -> CHAT
resp = await agent.resume_async(user_message=user_message)
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "CHAT"

# CHAT -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"
assert agent.fsm.state_bag["is_terminate_result"] is False
assert agent.fsm.state_bag["predicate_result"] is False


completed state: INIT

completed state: IS_TOOL_CALL

is_tool_call_result: False

predicate_result: False
I need to know your location to find the next public holiday for your area, as public holidays vary by country, state, or region.
Tool: "user_input"

completed state: CHAT

completed state: IS_TERMINATE


### b) This will throw error unless user input is provided

In [3]:
# IS_TERMINATE -> IS_TOOL_CALL
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TOOL_CALL"
assert agent.fsm.state_bag["is_tool_call_result"] is True
assert agent.fsm.state_bag["predicate_result"] is True
assert agent.fsm.state_bag["is_user_input"] is True

try:
    resp = await agent.resume_async()
except Exception as e:
    assert "pending user input" in str(e)
    print(f"Error occurred: {str(e)}")
    print("This is expected as the agent is waiting for user input.")



completed state: IS_TOOL_CALL
Error occurred: ToolUseAgent2.resume_async: pending user input
This is expected as the agent is waiting for user input.


### c) Can resume normally after user input

In [4]:
# IS_TOOL_CALL -> TOOL_USE
resp = await agent.resume_async("Use SGT")
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "TOOL_USE"
assert agent.fsm.state_bag["is_user_input"] is False

# TOOL_USE -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"

Thank you! I'll search for the next public holiday in Singapore (SGT timezone).
Tool: "search"
	Input: search_query = Singapore public holidays 2024 2025 next upcoming holiday dates

completed state: TOOL_USE

completed state: IS_TERMINATE


---

## Scenario 3: User interrupt agent with adhoc input

In this scenario, the user tries to distract the agent by interrupting it with an adhoc input.

### a) arrange


In [3]:
aggregated_client = McpAggregatedClient(["mcp-pseudo", "mcp-time", "mcp-web"])
agent = ToolUseAgent2(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
)

user_message = (
    "When is the next public holiday? Please ask if you need more information."
)

# INIT
print(f"\ncompleted state: {agent.fsm.state}")

# INIT -> IS_TOOL_CALL
resp = await agent.start_async()
assert agent.fsm.state == "IS_TOOL_CALL"
print(f"\ncompleted state: {agent.fsm.state}")
print(f"\nis_tool_call_result: {agent.fsm.state_bag['is_tool_call_result']}")
print(f"\npredicate_result: {agent.fsm.state_bag['predicate_result']}")

# IS_TOOL_CALL -> CHAT
resp = await agent.resume_async(user_message=user_message)
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "CHAT"

# CHAT -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"
assert agent.fsm.state_bag["is_terminate_result"] is False
assert agent.fsm.state_bag["predicate_result"] is False


completed state: INIT

completed state: IS_TOOL_CALL

is_tool_call_result: False

predicate_result: False
I need to know your location to find the next public holiday for your area, as public holidays vary by country and region.
Tool: "user_input"

completed state: CHAT

completed state: IS_TERMINATE


### b) User interrupt agent

Should be able to interrupt the agent and continue with original task.

In [4]:
# IS_TERMINATE -> IS_TOOL_CALL
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TOOL_CALL"
assert agent.fsm.state_bag["is_tool_call_result"] is True
assert agent.fsm.state_bag["predicate_result"] is True
assert agent.fsm.state_bag["is_user_input"] is True

# IS_TOOL_CALL -> TOOL_USE
resp = await agent.resume_async("Tell me a one paragraph joke")
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "TOOL_USE"

# TOOL_USE -> IS_TERMINATE
resp = await agent.resume_async()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"



completed state: IS_TOOL_CALL
I understand you'd like a joke, but I'm ToolUseAgent and I need to help you find information about the next public holiday first. To do that accurately, I need to know which country or region you're asking about, since public holidays vary significantly by location.
Tool: "user_input"

completed state: TOOL_USE

completed state: IS_TERMINATE
